### 1. Annotated

- `Annotated[타입, 메타데이터]` → 기존 타입에 부가정보(메타데이터)를 덧붙임
- 메타데이터(예: `"사용자 이름"`)는 런타임엔 무시됨 → 타입 검사 · 문서화 용도
- 타입 부분(`int` 등)은 그대로 적용 → 값이 어긋나면 IDE가 표시 (실행은 정상)

In [24]:
from typing import Annotated

# 메타데이터("...")는 설명용일 뿐 런타임엔 영향 없음
name: Annotated[str, "사용자 이름"]
age: Annotated[int, "사용자 나이"]

name = "홍길동"
age = "34"   # int인데 str 대입 → IDE 오류 표시 (런타임은 정상)

print(name)
print(age)

홍길동
34


### 2. Annotated + Pydantic

- Pydantic은 `Annotated`의 메타데이터(`Field`)를 **읽어서 실제 유효성 검사에 사용**
- 순수 타입 힌트와 달리 규칙 위반 시 **런타임에 `ValidationError` 발생**
- `Field(min_length=..., gt=..., ...)`로 길이 · 범위 등 제약을 지정

In [15]:
from typing import Annotated, List
from pydantic import Field, BaseModel, ValidationError

class Person(BaseModel):
    # ... 는 '필수'를 명시하는 표기 (기본값이 없으면 어차피 필수라 생략해도 동일)
    name: Annotated[str, Field(..., min_length=2, max_length=10, description="이름")]
    age: Annotated[int, Field(gt=18, lt=40, description="나이(19~39세)")]
    jobs: Annotated[List[str], Field(min_length=1, max_length=5, description="직업 갯수(1-5개)")]

### 3. 유효성 검사 성공

- 모든 제약을 만족 → 정상적으로 인스턴스 생성

In [11]:
try:
    person = Person(
        name="홍길동",
        age=34,
        jobs=["개발자"]
    )
    print("유효성 검사 성공:", person)
except ValidationError as err:
    print("유효성 검사 실패:", err)

유효성 검사 성공: name='홍길동' age=34 jobs=['개발자']


### 4. 유효성 검사 실패

- 하나라도 위반 시 `ValidationError` 발생 (위반 항목 전부 표시)
- 여기선 name 길이 초과 / age 범위 초과 / jobs 비어있음

In [6]:
try:
    person = Person(
        name="홍길동동무다리어카센타",
        age=40,
        jobs=[]
    )
    print("유효성 검사 성공:", person)
except ValidationError as err:
    print("유효성 검사 실패:", err)

유효성 검사 실패: 3 validation errors for Person
name
  String should have at most 10 characters [type=string_too_long, input_value='홍길동동무다리어카센타', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_too_long
age
  Input should be less than 40 [type=less_than, input_value=40, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than
jobs
  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/too_short


### 5. 필수 / 옵셔널 (Pydantic 필드 규칙)

- `Annotated` · `...` 와 무관하게 **기본값 유무로** 정해짐
- 기본값 없음 → 필수 / 기본값 있음 → 옵셔널
- `Optional[X]` 는 'None 허용'일 뿐 → 생략하려면 `= None` 필요

In [ ]:
from typing import Optional
from pydantic import BaseModel

class Member(BaseModel):
    name: str                       # 필수   (기본값 없음)
    age: int = 30                   # 옵셔널 (생략하면 30)
    nickname: Optional[str] = None  # 옵셔널 (생략하면 None)
    # 주의) Optional[str] 만 쓰고 '= None' 빼면 → 여전히 필수 (생략 시 'Field required' 에러)

print(
    Member(name="길동")
)   # age, nickname 은 생략 → 기본값 자동 채움